In [1]:
import pandas as pd
from lifelines import KaplanMeierFitter, WeibullAFTFitter, LogNormalAFTFitter



In [2]:
df = pd.read_csv("../data/trial_conversion.csv")
df.head()

,user_id,signup_date,signup_source,onboarding_completed,observed_through_date,days_to_conversion_or_censoring,converted
0,1,2024-04-24,referral,False,2024-06-01,4.9,1
1,2,2024-05-23,organic,False,2024-06-01,9.0,0
2,3,2024-04-15,referral,True,2024-06-01,7.5,1
3,4,2024-05-19,paid,False,2024-06-01,13.0,0
4,5,2024-05-27,organic,True,2024-06-01,5.0,0


In [3]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["observed_through_date"] = pd.to_datetime(df["observed_through_date"])

df["days_since_signup"] = (df["observed_through_date"] - df["signup_date"]).dt.days


df['onboarding_completed'] = df['onboarding_completed'].astype(int)

df.head()

,user_id,signup_date,signup_source,onboarding_completed,observed_through_date,days_to_conversion_or_censoring,converted,days_since_signup
0,1,2024-04-24,referral,0,2024-06-01,4.9,1,38
1,2,2024-05-23,organic,0,2024-06-01,9.0,0,9
2,3,2024-04-15,referral,1,2024-06-01,7.5,1,47
3,4,2024-05-19,paid,0,2024-06-01,13.0,0,13
4,5,2024-05-27,organic,1,2024-06-01,5.0,0,5


## Naive comparison
bucket the users into weekly signup cohorts,

In [4]:
df["signup_week"] = df["signup_date"].dt.to_period("W").apply(lambda p: p.start_time)

naive_by_cohort = df.groupby("signup_week").agg(
    n_users=("converted", "size"),
    naive_conversion_rate=("converted", "mean"),
    avg_days_since_signup=("days_since_signup", "mean"),
).reset_index()

naive_by_cohort

,signup_week,n_users,naive_conversion_rate,avg_days_since_signup
0,2024-03-04,32,0.968750,83.000000
1,2024-03-11,261,0.911877,79.157088
2,2024-03-18,238,0.924370,72.067227
3,2024-03-25,239,0.933054,64.991632
4,2024-04-01,268,0.940299,58.014925
5,2024-04-08,256,0.863281,50.921875
6,2024-04-15,258,0.945736,44.050388
7,2024-04-22,230,0.891304,36.695652
8,2024-04-29,277,0.906137,30.187726
9,2024-05-06,241,0.829876,22.746888


the most recent cohort has a lower conversion rate -- not because those users are less likely to convert, but because they did not have enough time. 

## Kaplan-meier per cohort


In [5]:


TRIAL_LENGTH_DAYS = 30

km_rows = []
for week, cohort in df.groupby("signup_week"):
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=cohort["days_to_conversion_or_censoring"],
        event_observed=cohort["converted"],
    )
    # S(30) = probability of NOT having converted by day 30
    survival_at_30 = kmf.survival_function_at_times(TRIAL_LENGTH_DAYS).iloc[0]
    km_rows.append(
        {
            "signup_week": week,
            "n_users": len(cohort),
            "max_followup_days": cohort["days_to_conversion_or_censoring"].max(),
            "km_projected_conversion_rate": 1 - survival_at_30,
        }
    )

km_by_cohort = pd.DataFrame(km_rows)

comparison = naive_by_cohort.merge(km_by_cohort, on=["signup_week", "n_users"])
comparison

,signup_week,n_users,naive_conversion_rate,avg_days_since_signup,max_followup_days,km_projected_conversion_rate
0,2024-03-04,32,0.968750,83.000000,28.2,1.000000
1,2024-03-11,261,0.911877,79.157088,30.0,0.929305
2,2024-03-18,238,0.924370,72.067227,30.0,0.935196
3,2024-03-25,239,0.933054,64.991632,30.0,0.944524
4,2024-04-01,268,0.940299,58.014925,30.0,0.956643
5,2024-04-08,256,0.863281,50.921875,30.0,0.888466
6,2024-04-15,258,0.945736,44.050388,30.0,0.960854
7,2024-04-22,230,0.891304,36.695652,30.0,0.907003
8,2024-04-29,277,0.906137,30.187726,30.0,0.921856
9,2024-05-06,241,0.829876,22.746888,26.0,0.878666


## Weibull AFT

In [6]:
dummies = pd.get_dummies(df["signup_source"], prefix="source", drop_first=True).astype(int)
model_df = pd.concat([df[["converted", "days_to_conversion_or_censoring",  "onboarding_completed"]], dummies], axis=1)

# WeibullAFTFitter requires strictly positive durations. 40 rows are 0 because
# those users signed up the same day as the snapshot -- zero follow-up time,
# not a data error. Nudge them up by half a day per lifelines' own suggestion.
model_df["days_to_conversion_or_censoring"] = model_df["days_to_conversion_or_censoring"].clip(lower=0.5)

In [7]:
MODEL_COLS = ['days_to_conversion_or_censoring', 'converted', 'onboarding_completed', 'source_paid', 'source_partner', 'source_referral']

In [8]:
weibull_aft = WeibullAFTFitter()

weibull_aft.fit(model_df, duration_col="days_to_conversion_or_censoring", event_col="converted")
weibull_aft.print_summary()

<lifelines.WeibullAFTFitter: fitted with 3000 total observations, 660 right-censored observations>
             duration col = 'days_to_conversion_or_censoring'
                event col = 'converted'
   number of observations = 3000
number of events observed = 2340
           log-likelihood = -8081.67
         time fit was run = 2026-07-27 23:55:09 UTC

---
                              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
param   covariate                                                                                                             
lambda_ onboarding_completed -0.49      0.61      0.02           -0.54           -0.45                0.58                0.64
        source_paid          -0.20      0.82      0.03           -0.25           -0.15                0.78                0.86
        source_partner       -0.40      0.67      0.04           -0.47           -0.33                0.62                0.72
        source_referral      -0.48      0.62      0.03           -0.54           -0.41                0.58                0.66
        Intercept             3.18     24.13      0.02            3.14            3.23               23.05               25.26
rho_    Intercept             0.60      1.82      0.02            0.57            0.63                1.76                1.88

                              cmp to      z      p  -log2(p)
param   covariate                                           
lambda_ onboarding_completed    0.00 -20.77 <0.005    315.91
        source_paid             0.00  -7.30 <0.005     41.61
        source_partner          0.00 -10.57 <0.005     84.29
        source_referral         0.00 -14.44 <0.005    154.67
        Intercept               0.00 136.28 <0.005       inf
rho_    Intercept               0.00  36.69 <0.005    976.83
---
Concordance = 0.65
AIC = 16175.34
log-likelihood ratio test = 632.73 on 4 df
-log2(p) of ll-ratio test = 448.11

In [9]:
lognormal_aft = LogNormalAFTFitter()
lognormal_aft.fit(model_df, duration_col="days_to_conversion_or_censoring", event_col="converted")

print("Weibull AIC:", weibull_aft.AIC_)
print("LogNormal AIC:", lognormal_aft.AIC_)

Weibull AIC: 16175.33846280484
LogNormal AIC: 16553.397593602287


## given a user hasn't converted by day 10, what's the probability they convert by day 30?

In [10]:
profile_onboarded = pd.DataFrame({
    "onboarding_completed": [1],
    "source_paid": [0],
    "source_partner": [0],
    "source_referral": [0],
})

profile_not_onboarded = pd.DataFrame({
    "onboarding_completed": [0],
    "source_paid": [0],
    "source_partner": [0],
    "source_referral": [0],
})

for label, profile in [("onboarded", profile_onboarded), ("not onboarded", profile_not_onboarded)]:
    surv = weibull_aft.predict_survival_function(profile, times=[10, 30])
    s10 = surv.loc[10].iloc[0]
    s30 = surv.loc[30].iloc[0]
    conditional_prob = 1 - s30 / s10
    print(f"{label}: S(10)={s10:.3f}, S(30)={s30:.3f}, P(convert by 30 | not converted by 10)={conditional_prob:.3f}")

onboarded: S(10)=0.610, S(30)=0.026, P(convert by 30 | not converted by 10)=0.957
not onboarded: S(10)=0.817, S(30)=0.226, P(convert by 30 | not converted by 10)=0.723
